# Chapter 10: Interpreting What ConvNets Learn (PyTorch Edition) — Version 2

This notebook is a **PyTorch conversion** of the companion notebook for [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition).

All Keras code has been converted to idiomatic PyTorch, with **enhanced visualizations** and **expanded explanations** to deepen understanding of ConvNet interpretability techniques.

**Version 2 Note**: This version includes **extensively detailed inline comments** on every code cell to make each function, variable, and operation crystal clear.

## What You'll Learn

| Technique | What It Reveals | Key Idea |
|---|---|---|
| **Intermediate Activations** | What features each layer detects | Forward-pass feature maps |
| **Filter Visualization** | What visual pattern each filter responds to | Gradient ascent on input space |
| **Grad-CAM** | Which image regions drive a classification | Gradient-weighted class activation maps |
| **t-SNE / PCA Embeddings** | How the network organizes its learned representations | Latent space geometry |

---

## 0. Setup & Imports

We install and import all required libraries upfront. We use **torchvision** for pretrained models and image transforms, and **matplotlib** for all visualizations.

In [ ]:
# ============================================================
# IMPORTS — All libraries needed throughout this notebook
# ============================================================

# --- Core PyTorch ---
import torch                           # Main PyTorch library for tensor operations
import torch.nn as nn                  # Neural network building blocks (layers, loss functions)
import torch.nn.functional as F        # Functional API (stateless operations like softmax, relu)

# --- Torchvision ---
import torchvision                     # Computer vision utilities (datasets, models, transforms)
import torchvision.transforms as transforms  # Image preprocessing pipelines
import torchvision.models as models          # Pretrained model zoo (VGG, ResNet, etc.)
from torchvision.models import VGG16_Weights, ResNet50_Weights  # Weight enums for pretrained models

# --- Data handling & visualization ---
import numpy as np                     # Numerical computing (arrays, math)
import matplotlib.pyplot as plt        # Plotting library for creating figures
import matplotlib.cm as cm             # Colormaps (e.g., 'jet', 'viridis') for heatmap visualization
from PIL import Image                  # Python Imaging Library — image loading/manipulation

# --- Networking & I/O ---
import requests                        # HTTP requests to download images from URLs
from io import BytesIO                 # In-memory binary stream — wraps raw bytes as a file-like object

# --- Misc ---
import warnings
warnings.filterwarnings("ignore")     # Suppress non-critical warnings for cleaner output

# ============================================================
# DEVICE SELECTION — Automatically pick the best available hardware
# Priority: CUDA GPU > Apple MPS (Metal) > CPU
# ============================================================
device = torch.device(
    "cuda" if torch.cuda.is_available() else          # NVIDIA GPU via CUDA
    "mps" if torch.backends.mps.is_available() else   # Apple Silicon GPU via Metal Performance Shaders
    "cpu"                                              # Fallback to CPU
)
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

In [ ]:
# ============================================================
# HELPER FUNCTIONS — Reusable utilities for the entire notebook
# ============================================================

def download_image(url, save_path=None):
    """
    Download an image from a URL and return it as a PIL Image.
    
    Args:
        url (str): The URL pointing to the image file.
        save_path (str, optional): If provided, save the image to this local path.
    
    Returns:
        PIL.Image.Image: The downloaded image converted to RGB format.
    """
    response = requests.get(url, timeout=30)  # HTTP GET with 30s timeout to avoid hanging
    # BytesIO wraps the raw bytes so PIL can read them as if they were a file on disk
    img = Image.open(BytesIO(response.content)).convert("RGB")  # Force RGB (drop alpha if present)
    if save_path:
        img.save(save_path)  # Optionally persist to disk
    return img


# --- ImageNet Preprocessing Pipeline ---
# VGG16 (and most ImageNet-trained models) expect:
#   1. Input size: 224×224 pixels
#   2. Pixel values as tensors in [0, 1]
#   3. Channel-wise normalization using ImageNet training set statistics
imagenet_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),    # Resize any image to the expected 224×224 input size
    transforms.ToTensor(),            # Convert PIL Image (H,W,C) uint8 [0,255] → Tensor (C,H,W) float [0,1]
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],   # Per-channel mean of the ImageNet training set (R, G, B)
        std=[0.229, 0.224, 0.225],    # Per-channel std of the ImageNet training set (R, G, B)
    ),  # After normalization, pixel values are roughly in [-2, 2]
])


def denormalize(tensor):
    """
    Reverse ImageNet normalization so we can display the tensor as a natural image.
    
    The normalization formula is:  normalized = (pixel - mean) / std
    So to reverse it:             pixel = normalized * std + mean
    
    Args:
        tensor (torch.Tensor): A normalized image tensor of shape (3, H, W).
    
    Returns:
        torch.Tensor: Denormalized tensor with values clamped to [0, 1] for display.
    """
    # Reshape mean & std to (3, 1, 1) so they broadcast across spatial dimensions (H, W)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)
    tensor = tensor.cpu().clone()    # Work on a CPU copy to avoid modifying the original
    tensor = tensor * std + mean     # Reverse the normalization: x_original = x_normalized * std + mean
    tensor = tensor.clamp(0, 1)      # Clamp to valid display range [0, 1]
    return tensor


print("✅ Helpers loaded.")

---
## 1. Understanding the Model Architecture

Before we interpret what a ConvNet learns, we need to understand its structure. We'll use **VGG16** — a well-known architecture with a clean, sequential stack of convolutional blocks, making it ideal for visualization.

### VGG16 Architecture Diagram

```
Input (224×224×3)
  │
  ├─ Block 1: Conv3-64 → Conv3-64 → MaxPool
  ├─ Block 2: Conv3-128 → Conv3-128 → MaxPool
  ├─ Block 3: Conv3-256 → Conv3-256 → Conv3-256 → MaxPool
  ├─ Block 4: Conv3-512 → Conv3-512 → Conv3-512 → MaxPool
  ├─ Block 5: Conv3-512 → Conv3-512 → Conv3-512 → MaxPool
  │
  ├─ Flatten
  ├─ FC-4096 → FC-4096 → FC-1000
  │
  Output (1000 classes)
```

**Key insight**: Earlier layers detect low-level features (edges, textures), while deeper layers capture high-level semantic concepts (faces, objects).

In [ ]:
# ============================================================
# LOAD PRETRAINED VGG16 — Trained on ImageNet (1.2M images, 1000 classes)
# ============================================================

# models.vgg16() creates the VGG16 architecture.
# weights=VGG16_Weights.IMAGENET1K_V1 loads weights pretrained on ImageNet.
# .to(device) moves all parameters (weights, biases) to the selected hardware (GPU/CPU).
model = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).to(device)

# .eval() sets the model to evaluation mode:
#   - Disables dropout (which randomly zeros neurons during training)
#   - Switches BatchNorm to use running statistics instead of batch statistics
#   - Does NOT disable gradient computation (use torch.no_grad() for that)
model.eval()

# --- Print the full architecture ---
# VGG16 has two main sub-modules:
#   model.features  — the convolutional feature extractor (Conv2d + ReLU + MaxPool layers)
#   model.classifier — the fully connected classification head (Linear + ReLU + Dropout)
print("=" * 70)
print("VGG16 Architecture")
print("=" * 70)

# Iterate over each layer in the feature extractor and print its index + type
# named_children() yields (name_string, module) pairs for direct children only
for name, module in model.features.named_children():
    print(f"  features.{name:>2s}: {module}")

print("  --- AdaptiveAvgPool2d ---")  # Sits between features and classifier

# Iterate over each layer in the classifier head
for name, module in model.classifier.named_children():
    print(f"  classifier.{name}: {module}")

print("=" * 70)

In [ ]:
# ============================================================
# VISUALIZE PARAMETER COUNTS — Where does the model store its knowledge?
# ============================================================

# --- Collect parameter counts for each convolutional layer ---
layer_names = []     # Human-readable labels for each layer
param_counts = []    # Number of learnable parameters (weights + biases) per layer
layer_types = []     # Layer class name (Conv2d, etc.) for color-coding

for name, module in model.features.named_children():
    # p.numel() returns the total number of elements in a parameter tensor
    # e.g., a Conv2d with 64 filters of size 3×3×3 has weight shape (64,3,3,3) → 1728 params + 64 biases
    n_params = sum(p.numel() for p in module.parameters())
    if n_params > 0:  # Skip ReLU and MaxPool (they have zero learnable parameters)
        layer_names.append(f"features.{name}\n{module.__class__.__name__}")
        param_counts.append(n_params)
        layer_types.append(module.__class__.__name__)

# Color-code: blue for Conv2d, orange for anything else
colors = ['#2196F3' if 'Conv' in t else '#FF9800' for t in layer_types]

# --- Create a side-by-side figure: bar chart + pie chart ---
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# LEFT: Horizontal bar chart — parameter count per convolutional layer
axes[0].barh(range(len(layer_names)), param_counts, color=colors, edgecolor='white')
axes[0].set_yticks(range(len(layer_names)))          # Position tick marks
axes[0].set_yticklabels(layer_names, fontsize=8)     # Label each bar
axes[0].set_xlabel("Number of Parameters", fontsize=12)
axes[0].set_title("Parameters per Convolutional Layer", fontsize=14, fontweight='bold')
axes[0].invert_yaxis()  # Put the first layer at the top
# Annotate each bar with the exact count
for i, v in enumerate(param_counts):
    axes[0].text(v + 1000, i, f"{v:,}", va='center', fontsize=7)

# RIGHT: Pie chart — feature extractor vs. classifier parameter split
# This reveals that most parameters live in the FC classifier, not the conv layers!
feat_params = sum(p.numel() for p in model.features.parameters())      # ~14.7M
class_params = sum(p.numel() for p in model.classifier.parameters())   # ~123.6M
axes[1].pie(
    [feat_params, class_params],
    labels=[f"Features\n{feat_params:,}", f"Classifier\n{class_params:,}"],
    colors=['#2196F3', '#E91E63'],
    autopct='%1.1f%%',      # Show percentage on each slice
    startangle=90,           # Rotate so the first slice starts at 12 o'clock
    textprops={'fontsize': 11}
)
axes[1].set_title("Feature Extractor vs Classifier Parameters", fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig("vgg16_architecture_params.png", dpi=150, bbox_inches='tight')  # Save high-res copy
plt.show()

# Print the grand total
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

---
## 2. Load & Display a Test Image

We'll download a cat image (same as the book) and preprocess it for VGG16. Let's also visualize what the preprocessing does to the image.

In [ ]:
# ============================================================
# DOWNLOAD AND PREPROCESS A TEST IMAGE
# ============================================================

# Download the cat image used in the book
cat_url = "https://img-datasets.s3.amazonaws.com/cat.jpg"
cat_img = download_image(cat_url, save_path="cat.jpg")  # Returns a PIL Image in RGB

# --- Preprocess for VGG16 ---
# imagenet_preprocess: Resize(224) → ToTensor() → Normalize(ImageNet stats)
# .unsqueeze(0) adds a batch dimension: (3, 224, 224) → (1, 3, 224, 224)
# Models expect a batch dimension even for a single image
# .to(device) moves the tensor to GPU/CPU to match the model's device
img_tensor = imagenet_preprocess(cat_img).unsqueeze(0).to(device)  # Shape: (1, 3, 224, 224)

# --- Create a simpler transform for display purposes (NO normalization) ---
# This lets us show the resized image without the color distortion from normalization
display_transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to match model input
    transforms.ToTensor(),          # Convert to tensor [0, 1] but do NOT normalize
])
img_display = display_transform(cat_img)  # Shape: (3, 224, 224), values in [0, 1]

# ============================================================
# VISUALIZE THE PREPROCESSING PIPELINE
# Show: Original → Resized → Normalized
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Panel 1: Original image (arbitrary size, as-downloaded)
axes[0].imshow(cat_img)  # PIL images can be displayed directly by matplotlib
axes[0].set_title("Original Image", fontsize=14, fontweight='bold')
axes[0].axis("off")  # Hide axis ticks and labels

# Panel 2: Resized to 224×224, no normalization
# .permute(1, 2, 0) rearranges from (C, H, W) → (H, W, C) which matplotlib expects
axes[1].imshow(img_display.permute(1, 2, 0))
axes[1].set_title("Resized (224×224)", fontsize=14, fontweight='bold')
axes[1].axis("off")

# Panel 3: After ImageNet normalization (denormalized back to [0,1] for display)
# Without denormalization, the image would look weird because normalized values
# can be negative (which matplotlib clips to 0 → darkens the image)
normalized_display = denormalize(img_tensor[0])  # Remove batch dim [0], then denormalize
axes[2].imshow(normalized_display.permute(1, 2, 0))
axes[2].set_title("ImageNet Normalized", fontsize=14, fontweight='bold')
axes[2].axis("off")

plt.suptitle("Image Preprocessing Pipeline", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# Print tensor metadata to verify preprocessing
print(f"Input tensor shape: {img_tensor.shape}")  # Should be [1, 3, 224, 224]
print(f"Input tensor range: [{img_tensor.min():.2f}, {img_tensor.max():.2f}]")  # Roughly [-2, 2]

---
## 3. Visualizing Intermediate Activations

**Concept**: Each convolutional layer transforms the input into a set of *feature maps* (also called *activation maps*). By visualizing these maps, we can see **what patterns each layer detects**.

- **Early layers** → edges, colors, simple textures  
- **Middle layers** → parts of objects (eyes, ears, fur patterns)  
- **Deep layers** → high-level abstract concepts  

In PyTorch, we use **forward hooks** to capture intermediate outputs — this is more flexible than Keras's functional API approach.

In [ ]:
# ============================================================
# EXTRACT INTERMEDIATE ACTIVATIONS USING FORWARD HOOKS
# ============================================================
#
# WHAT IS A HOOK?
# A "hook" is a callback function that PyTorch calls automatically during
# the forward (or backward) pass. A forward hook is called right after
# a module computes its output, giving us access to that output.
#
# WHY HOOKS?
# VGG16 is a Sequential model — we can't easily access intermediate
# layer outputs without hooks. Hooks let us "tap into" any layer
# without modifying the model code.
#
# HOOK SIGNATURE: hook(module, input, output)
#   - module: the layer itself (e.g., Conv2d)
#   - input:  tuple of input tensors to the layer
#   - output: the layer's output tensor (what we want to capture)
# ============================================================

# Dictionary to store captured activations, keyed by layer name
activations = {}

def get_activation_hook(name):
    """
    Factory function that creates a hook for a specific layer.
    
    We need this factory pattern (a function that returns a function) because
    each hook needs its own 'name' variable captured via closure. If we just
    used a single function, all hooks would share the same 'name' reference
    and overwrite each other.
    
    Args:
        name (str): A label for this layer, used as the dictionary key.
    
    Returns:
        callable: A hook function that stores the layer's output in `activations[name]`.
    """
    def hook(module, input, output):
        # .detach() disconnects the tensor from the computation graph (saves memory)
        # .cpu() moves it to CPU so we can work with it in numpy later
        activations[name] = output.detach().cpu()
    return hook


# --- Register hooks on all Conv2d and MaxPool2d layers ---
hooks = []  # Keep references to hooks so we can remove them later

for name, module in model.features.named_children():
    # Only attach hooks to Conv2d and MaxPool2d layers (skip ReLU — it's in-place)
    if isinstance(module, (nn.Conv2d, nn.MaxPool2d)):
        # Create a descriptive label like "features.0 (Conv2d)"
        layer_label = f"features.{name} ({module.__class__.__name__})"
        # register_forward_hook() returns a handle we can use to remove the hook later
        h = module.register_forward_hook(get_activation_hook(layer_label))
        hooks.append(h)


# --- Forward pass to trigger all hooks and capture activations ---
# torch.no_grad() disables gradient computation (faster + less memory)
# We don't need gradients here — we're just running inference
with torch.no_grad():
    _ = model(img_tensor)  # The output is discarded; we care about the hook side-effects

# Print summary of captured activations
print(f"✅ Captured activations from {len(activations)} layers:")
for name, act in activations.items():
    # Each activation has shape (batch, channels, height, width)
    # e.g., features.0 (Conv2d) → [1, 64, 224, 224] = 64 feature maps of size 224×224
    print(f"  {name:45s} → shape: {list(act.shape)}")

# --- IMPORTANT: Remove hooks to prevent memory leaks ---
# If hooks are left attached, they'll fire on every future forward pass,
# accumulating stored tensors and slowing down the model.
for h in hooks:
    h.remove()

### 3.1 First Layer Activations — Edge & Color Detectors

The very first convolutional layer has 64 filters of size 3×3. Each filter responds to a different low-level pattern. Let's visualize individual channels of the first Conv2d activation.

In [ ]:
# ============================================================
# VISUALIZE THE FIRST CONVOLUTIONAL LAYER'S FEATURE MAPS
# ============================================================

# Get the activation from the very first layer in our dictionary
first_layer_name = list(activations.keys())[0]  # e.g., "features.0 (Conv2d)"
first_act = activations[first_layer_name][0]     # Remove batch dim: (1, 64, 224, 224) → (64, 224, 224)

print(f"First layer: {first_layer_name}")
print(f"Activation shape: {first_act.shape}  (channels, height, width)")
# 64 channels = 64 different feature maps, each 224×224 pixels
# Each channel shows how strongly a particular filter responded at each spatial location

# --- Plot the first 32 channels in a 4×8 grid ---
fig, axes = plt.subplots(4, 8, figsize=(20, 10))  # 4 rows × 8 columns = 32 subplots
fig.suptitle(f"First Layer Feature Maps — {first_layer_name}",
             fontsize=16, fontweight='bold', y=1.02)

# axes.flat flattens the 2D array of axes into a 1D iterator for easy enumeration
for i, ax in enumerate(axes.flat):
    if i < first_act.shape[0]:  # Safety check (we have 64 channels but only 32 subplots)
        # Display one channel as a 2D heatmap
        # .numpy() converts the PyTorch tensor to a numpy array (required by matplotlib)
        # cmap='viridis' uses a perceptually-uniform colormap (dark purple → yellow)
        ax.imshow(first_act[i].numpy(), cmap='viridis')
        ax.set_title(f"Ch {i}", fontsize=9)  # Label with the channel index
    ax.axis("off")  # Hide axis ticks/labels for a clean look

plt.tight_layout()
plt.show()

### 3.2 Activations Across All Layers

Now let's visualize feature maps from **every convolutional layer**, showing how the representation evolves from concrete pixels to abstract concepts.

In [ ]:
# ============================================================
# VISUALIZE ACTIVATIONS FROM ALL CONV2D LAYERS
# Creates a tiled grid of feature map thumbnails for each layer
# ============================================================

images_per_row = 16  # Number of feature map thumbnails per row in the grid

# Filter to keep only Conv2d layers (skip MaxPool2d layers)
conv_layers = {k: v for k, v in activations.items() if "Conv2d" in k}

for layer_name, layer_activation in conv_layers.items():
    act = layer_activation[0]   # Remove batch dim: (1, C, H, W) → (C, H, W)
    n_features = act.shape[0]   # Number of channels (feature maps) — e.g., 64, 128, 256, 512
    size = act.shape[1]         # Spatial dimension (H = W since input is square)
    
    # Calculate how many rows of thumbnails to show (cap at 8 for readability)
    n_cols = min(n_features // images_per_row, 8)
    if n_cols == 0:
        n_cols = 1  # Ensure at least one row

    # --- Create a blank canvas to stitch thumbnails into ---
    # Each thumbnail is `size × size` pixels, with 1-pixel gaps between them
    display_grid = np.zeros((
        (size + 1) * n_cols - 1,           # Total height: n_cols rows of thumbnails + gaps
        images_per_row * (size + 1) - 1     # Total width: images_per_row columns + gaps
    ))

    # --- Fill the grid with normalized feature map images ---
    for col in range(n_cols):           # Iterate over rows in the grid
        for row in range(images_per_row):  # Iterate over columns in the grid
            channel_index = col * images_per_row + row  # Linear index into the channels
            if channel_index >= n_features:
                break  # No more channels to display

            channel_image = act[channel_index].numpy().copy()  # Extract one feature map

            # Normalize the feature map for display:
            # Subtract mean, divide by std, scale to roughly [0, 255]
            if channel_image.std() != 0:  # Avoid division by zero for dead channels
                channel_image -= channel_image.mean()     # Center around 0
                channel_image /= channel_image.std()      # Normalize to unit variance
                channel_image *= 64                        # Scale up
                channel_image += 128                       # Shift to center around 128
            channel_image = np.clip(channel_image, 0, 255).astype("uint8")  # Clamp to valid uint8 range

            # Place the thumbnail into the correct position in the grid
            display_grid[
                col * (size + 1): (col + 1) * size + col,  # y-slice (row position)
                row * (size + 1): (row + 1) * size + row   # x-slice (column position)
            ] = channel_image

    # --- Display the stitched grid ---
    scale = 1.0 / size  # Scale figure size relative to thumbnail size
    plt.figure(figsize=(scale * display_grid.shape[1],
                        scale * display_grid.shape[0]))
    plt.title(layer_name, fontsize=14, fontweight='bold')
    plt.grid(False)    # Disable matplotlib's grid lines
    plt.axis("off")
    plt.imshow(display_grid, aspect="auto", cmap="viridis")
    plt.show()

### 3.3 Activation Statistics Across Layers (Added Visualization)

Let's quantify how activations change across layers — measuring **sparsity** (fraction of dead neurons), **mean activation**, and **standard deviation**. This reveals how the network progressively "focuses" on fewer, more specialized features.

In [ ]:
# ============================================================
# COMPUTE AND PLOT ACTIVATION STATISTICS PER LAYER
# ============================================================
# This analysis quantifies HOW activations change with depth.
# Key metrics:
#   - Mean activation: average response magnitude
#   - Std deviation:   how varied the responses are
#   - Sparsity:        % of neurons that output exactly 0 (killed by ReLU)
#   - Spatial size:    how the feature map resolution decreases through MaxPool layers

layer_stats = []

for name, act in activations.items():
    if "Conv2d" not in name:
        continue  # Skip MaxPool layers — we only want Conv2d stats
    
    a = act[0].numpy()  # Shape: (C, H, W) — remove batch dimension and convert to numpy
    
    layer_stats.append({
        'name': name.split('(')[0].strip(),  # Extract clean name, e.g., "features.0"
        'mean': a.mean(),                     # Global mean across all channels and pixels
        'std': a.std(),                       # Global standard deviation
        'sparsity': (a == 0).mean() * 100,    # % of values that are exactly 0 (after ReLU)
        'max': a.max(),                       # Maximum activation value
        'n_channels': a.shape[0],             # Number of feature maps (filters) in this layer
        'spatial': a.shape[1],                # Spatial resolution (H = W)
    })

# --- Create a 2×2 grid of bar charts ---
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Activation Statistics Across VGG16 Layers",
             fontsize=16, fontweight='bold')

x = range(len(layer_stats))  # x-axis positions (one per Conv2d layer)
# Shorten names for x-axis labels ("features.0" → "f.0")
short_names = [s['name'].replace('features.', 'f.') for s in layer_stats]

# TOP-LEFT: Mean activation per layer
# Deeper layers tend to have lower mean → more selective responses
axes[0, 0].bar(x, [s['mean'] for s in layer_stats], color='#2196F3', edgecolor='white')
axes[0, 0].set_title("Mean Activation", fontweight='bold')
axes[0, 0].set_xticks(x)
axes[0, 0].set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)

# TOP-RIGHT: Standard deviation of activations
# Higher std → more dynamic range → the layer is "doing more work"
axes[0, 1].bar(x, [s['std'] for s in layer_stats], color='#FF9800', edgecolor='white')
axes[0, 1].set_title("Activation Std Dev", fontweight='bold')
axes[0, 1].set_xticks(x)
axes[0, 1].set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)

# BOTTOM-LEFT: Sparsity (% of zero-valued activations after ReLU)
# ReLU(x) = max(0, x) — any negative activation becomes 0
# Higher sparsity in deeper layers = the network is more selective,
# only "firing" for very specific features
axes[1, 0].bar(x, [s['sparsity'] for s in layer_stats], color='#E91E63', edgecolor='white')
axes[1, 0].set_title("Sparsity (% Zero Activations after ReLU)", fontweight='bold')
axes[1, 0].set_ylabel("%")
axes[1, 0].set_xticks(x)
axes[1, 0].set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)

# BOTTOM-RIGHT: Spatial resolution per layer
# Each MaxPool2d halves the resolution: 224 → 112 → 56 → 28 → 14 → 7
# Smaller spatial size = each neuron's receptive field covers more of the original image
axes[1, 1].bar(x, [s['spatial'] for s in layer_stats], color='#4CAF50', edgecolor='white')
axes[1, 1].set_title("Spatial Resolution (H=W)", fontweight='bold')
axes[1, 1].set_ylabel("Pixels")
axes[1, 1].set_xticks(x)
axes[1, 1].set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)

plt.tight_layout()
plt.show()

print("\n📊 Key Observation: Sparsity increases in deeper layers → the network")
print("   becomes more selective, activating only for specific high-level features.")

### 3.4 Visualizing Convolutional Filters (Weights) Directly

Before gradient ascent, let's look at what the **raw learned weights** of the first convolutional layer look like. Since the first layer operates on RGB input, we can directly visualize its 3×3×3 filters as tiny color images.

In [ ]:
# ============================================================
# VISUALIZE FIRST-LAYER CONVOLUTIONAL FILTER WEIGHTS
# ============================================================
# The first Conv2d layer is special because its input is a 3-channel RGB image.
# Each of its 64 filters has shape (3, 3, 3) = (in_channels=3, height=3, width=3).
# Since input_channels=3 corresponds to R,G,B, we can display each filter as
# a tiny 3×3 color image — this shows what color/edge pattern the filter detects.
#
# For deeper layers, filters operate on abstract feature maps (not RGB),
# so we can't visualize them as color images — that's why we need gradient ascent.

first_conv = model.features[0]             # First Conv2d layer in VGG16
filters = first_conv.weight.data.cpu().clone()  # Shape: (64, 3, 3, 3) — 64 filters, 3×3×3 each
# .data accesses the raw tensor (without autograd metadata)
# .cpu().clone() makes a detached CPU copy so we don't accidentally modify model weights

# --- Normalize each filter independently to [0, 1] for display ---
# Each filter has different value ranges, so we min-max normalize per filter
filters_min = filters.view(64, -1).min(dim=1)[0].view(64, 1, 1, 1)  # Per-filter minimum
# .view(64, -1) reshapes (64, 3, 3, 3) → (64, 27) to find min across all 27 values per filter
# .min(dim=1)[0] returns the min values (index [0] of the (values, indices) tuple)
# .view(64, 1, 1, 1) reshapes back for broadcasting
filters_max = filters.view(64, -1).max(dim=1)[0].view(64, 1, 1, 1)  # Per-filter maximum
filters_norm = (filters - filters_min) / (filters_max - filters_min + 1e-8)  # Normalize to [0, 1]
# The 1e-8 prevents division by zero if a filter is all-constant

# --- Plot all 64 filters in an 8×8 grid ---
fig, axes = plt.subplots(8, 8, figsize=(14, 14))
fig.suptitle("First Conv Layer Filters (3×3×3 → visualized as RGB)",
             fontsize=16, fontweight='bold', y=1.01)

for i, ax in enumerate(axes.flat):
    # Convert from PyTorch format (C, H, W) → matplotlib format (H, W, C)
    f = filters_norm[i].permute(1, 2, 0).numpy()  # Shape: (3, 3, 3) → (3, 3, 3) but H,W,C order
    
    # Upscale the tiny 3×3 image by 20× using nearest-neighbor (repeat pixels)
    # so it's visible — a 3×3 image would be invisible at normal figure size
    f_upscaled = np.repeat(np.repeat(f, 20, axis=0), 20, axis=1)  # (60, 60, 3)
    ax.imshow(f_upscaled)
    ax.set_title(f"Filter {i}", fontsize=7)
    ax.axis("off")

plt.tight_layout()
plt.show()
print("🔍 These tiny 3×3 filters detect edges, color gradients, and basic textures.")

---
## 4. Visualizing ConvNet Filters via Gradient Ascent

**Concept**: To understand what a filter "looks for", we can generate an input image that **maximally activates** a given filter. We start from random noise and iteratively adjust the image using **gradient ascent** to increase the filter's response.

$$\text{image}_{t+1} = \text{image}_t + \eta \cdot \frac{\partial \text{activation}_{filter}}{\partial \text{image}_t}$$

This is the reverse of training: instead of updating weights, we update the **input image** while keeping weights frozen.

In [ ]:
# ============================================================
# GRADIENT ASCENT FOR FILTER VISUALIZATION
# ============================================================

class FeatureExtractor(nn.Module):
    """
    A wrapper that extracts features from a specific layer of VGG16.
    
    Instead of running the full model (which includes the classifier),
    this module only runs the feature extractor up to (and including)
    the target layer. This is used for gradient ascent: we need to
    compute gradients of a specific layer's output w.r.t. the input image.
    
    Args:
        model: The full VGG16 model.
        target_layer_idx (int): Index of the last layer to include
            (e.g., 17 for features[0:18]).
    """
    def __init__(self, model, target_layer_idx):
        super().__init__()
        # Slice the feature extractor: take only layers 0 through target_layer_idx (inclusive)
        # list(model.features.children()) gets all child modules as a list
        # [:target_layer_idx + 1] slices up to and including the target
        # nn.Sequential(*...) wraps them back into a sequential model
        self.features = nn.Sequential(*list(model.features.children())[:target_layer_idx + 1])

    def forward(self, x):
        """Run input through the truncated model and return feature maps."""
        return self.features(x)


def deprocess_image(img_tensor):
    """
    Convert an optimized tensor (from gradient ascent) back to a displayable
    numpy image in uint8 [0, 255] format.
    
    The gradient ascent process produces tensors with arbitrary value ranges.
    This function normalizes them to a visually meaningful [0, 255] range.
    
    Args:
        img_tensor (torch.Tensor): Shape (1, 3, H, W) — the optimized image.
    
    Returns:
        np.ndarray: Shape (H, W, 3), dtype uint8, values in [0, 255].
    """
    img = img_tensor.cpu().clone().squeeze(0)  # Remove batch dim: (1, 3, H, W) → (3, H, W)
    img = img.permute(1, 2, 0).numpy()         # (3, H, W) → (H, W, 3) for matplotlib
    # Normalize to zero mean and unit variance, then scale to [0, 1]
    img -= img.mean()            # Center around 0
    img /= (img.std() + 1e-5)    # Normalize to unit variance (1e-5 prevents division by zero)
    img *= 0.25                   # Scale to roughly [-0.5, 0.5]
    img += 0.5                    # Shift to roughly [0, 1]
    img = np.clip(img, 0, 1)      # Hard-clip to [0, 1]
    img = (img * 255).astype(np.uint8)  # Scale to [0, 255] and convert to integer
    return img


def generate_filter_pattern(model, layer_idx, filter_index,
                             img_size=128, iterations=30, lr=1.0):
    """
    Generate an image that maximally activates a specific filter
    using gradient ascent on the input pixels.
    
    Algorithm:
        1. Start with a random noise image
        2. Forward-pass through the model up to the target layer
        3. Compute a loss = mean activation of the target filter
        4. Backpropagate to get gradients w.r.t. the input image
        5. Update the image in the direction that INCREASES the activation
           (gradient ASCENT, not descent)
        6. Repeat for `iterations` steps
    
    Args:
        model: The full VGG16 model (weights are frozen; only the image is updated).
        layer_idx (int): Index of the target convolutional layer in model.features.
        filter_index (int): Which filter (channel) to maximize in the target layer.
        img_size (int): Height and width of the generated image (default 128).
        iterations (int): Number of gradient ascent steps (default 30).
        lr (float): Learning rate / step size for the gradient update (default 1.0).
    
    Returns:
        torch.Tensor: The optimized image tensor of shape (1, 3, img_size, img_size).
    """
    # Build a truncated model that outputs features at the target layer
    extractor = FeatureExtractor(model, layer_idx).to(device)
    extractor.eval()  # Set to eval mode (affects BatchNorm/Dropout, not relevant for VGG but good practice)

    # --- Initialize with random noise ---
    # Small random values centered around 0.5 (gray) with tiny perturbations
    # The image is our "learnable parameter" — we'll optimize it via gradients
    img = torch.randn(1, 3, img_size, img_size, device=device) * 0.02 + 0.5
    img.requires_grad_(True)  # Enable gradient tracking on the image tensor

    for i in range(iterations):
        # --- Forward pass: get the target layer's activation ---
        activation = extractor(img)  # Shape: (1, n_filters, H', W')

        # --- Compute the loss: mean activation of the target filter ---
        # activation[0, filter_index] selects one feature map: (H', W')
        # [2:-2, 2:-2] trims 2 pixels from each border to avoid edge artifacts
        # (border pixels can have boundary effects from convolutions)
        # .mean() computes the average activation — this is what we want to MAXIMIZE
        loss = activation[0, filter_index, 2:-2, 2:-2].mean()

        # --- Backward pass: compute gradients of loss w.r.t. the input image ---
        loss.backward()

        # --- Normalize the gradients ---
        # Dividing by the L2 norm ensures that the step size is consistent
        # regardless of the gradient magnitude. Without this, some iterations
        # might take huge steps while others barely move.
        grads = img.grad.data                 # Access the computed gradients
        grads /= (grads.norm() + 1e-8)        # L2-normalize (1e-8 prevents div-by-zero)

        # --- Gradient ASCENT step ---
        # Unlike training (where we subtract gradients to minimize loss),
        # here we ADD gradients to maximize activation.
        # img.data += ... modifies the tensor in-place without affecting the computation graph
        img.data += lr * grads

        # --- Zero out the gradients for the next iteration ---
        # PyTorch accumulates gradients by default; we must clear them manually
        img.grad.data.zero_()

    # .detach() removes the tensor from the computation graph (we're done optimizing)
    return img.detach()


print("✅ Gradient ascent functions defined.")

In [ ]:
# ============================================================
# DEMO: Visualize a single filter pattern from a mid-level layer
# ============================================================

# VGG16's features[17] is a Conv2d layer in Block 3 (256 filters)
# This is a "mid-level" layer that detects textures and simple shapes
layer_idx = 17
filter_idx = 2   # Arbitrary filter choice — try different values!

# Run gradient ascent to generate the pattern that maximally excites this filter
pattern = generate_filter_pattern(
    model, layer_idx, filter_idx,
    img_size=150,     # Generate a 150×150 pixel image
    iterations=40,    # 40 optimization steps
    lr=1.0            # Step size
)
# Convert the raw tensor to a displayable image
img = deprocess_image(pattern)

# --- Plot the result alongside an explanation ---
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# LEFT: The generated filter pattern
axes[0].imshow(img)
axes[0].set_title(f"Filter {filter_idx} Pattern\n(features[{layer_idx}])",
                  fontsize=13, fontweight='bold')
axes[0].axis("off")

# RIGHT: Textual explanation of the gradient ascent process
axes[1].text(0.5, 0.5,
    "Gradient Ascent Process:\n\n"
    "1. Start with random noise\n"
    "2. Forward pass → get filter activation\n"
    "3. Compute ∂activation/∂image\n"
    "4. Update image += lr × normalized_grad\n"
    "5. Repeat 30-40 times\n\n"
    "Result: the image that this filter\n"
    "\"dreams\" about seeing",
    transform=axes[1].transAxes,  # Use axes-relative coordinates (0-1 range)
    fontsize=12,
    verticalalignment='center', horizontalalignment='center',
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
axes[1].axis("off")
axes[1].set_title("How It Works", fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

### 4.1 Filter Patterns Across Multiple Layers

Let's generate filter visualizations for **multiple layers** to see how patterns evolve from simple edges to complex textures.

In [ ]:
# ============================================================
# GENERATE FILTER PATTERNS FOR SELECTED LAYERS
# Shows how learned features evolve from simple → complex with depth
# ============================================================

# Map of layer indices to descriptive names
# VGG16 Conv2d layer indices within model.features:
#   Block 1: 0, 2    Block 2: 5, 7    Block 3: 10, 12, 14
#   Block 4: 17, 19, 21    Block 5: 24, 26, 28
# We pick the first Conv2d from each block to see the progression
target_layers = {
    0:  "Block 1 - Conv1 (64 filters)",    # Earliest: detects edges, colors
    5:  "Block 2 - Conv1 (128 filters)",   # Simple textures
    10: "Block 3 - Conv1 (256 filters)",   # More complex textures
    17: "Block 4 - Conv1 (512 filters)",   # Object parts
    24: "Block 5 - Conv1 (512 filters)",   # High-level abstract features
}

n_filters_to_show = 8  # Number of filters to visualize per layer

# Create a grid: rows = layers, columns = individual filters
fig, axes = plt.subplots(len(target_layers), n_filters_to_show,
                          figsize=(20, 3 * len(target_layers)))
fig.suptitle("Filter Patterns Across VGG16 Layers\n(Early → Deep)",
             fontsize=18, fontweight='bold', y=1.02)

for row, (layer_idx, layer_desc) in enumerate(target_layers.items()):
    for col in range(n_filters_to_show):
        # Progress indicator (\r overwrites the same line)
        print(f"\r  Generating: {layer_desc}, filter {col+1}/{n_filters_to_show}", end="")

        # Generate the pattern image for this specific layer + filter
        pattern = generate_filter_pattern(
            model, layer_idx, col,  # col = filter_index
            img_size=128,           # 128×128 pixel output
            iterations=30,          # 30 gradient ascent steps
            lr=1.0
        )
        img = deprocess_image(pattern)  # Convert to displayable numpy array

        # Plot the pattern in the grid
        axes[row, col].imshow(img)
        axes[row, col].axis("off")

        # Add row labels on the leftmost column
        if col == 0:
            axes[row, col].set_ylabel(
                layer_desc, fontsize=10, fontweight='bold',
                rotation=0,          # Horizontal text
                labelpad=150,        # Distance from the plot
                va='center'          # Vertical alignment
            )
        # Add column headers on the top row
        if row == 0:
            axes[row, col].set_title(f"Filter {col}", fontsize=10)

print("\n✅ Done!")
plt.tight_layout()
plt.savefig("filter_patterns_across_layers.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Observation:")
print("  • Block 1: Simple edges and color gradients")
print("  • Block 2-3: Textures and simple patterns")
print("  • Block 4-5: Complex, abstract textures and object parts")

### 4.2 Grid of 64 Filters from a Single Layer

Let's create a dense 8×8 grid showing all 64 filter patterns from a single block, similar to the book's figure.

In [ ]:
# ============================================================
# GENERATE ALL 64 FILTER PATTERNS FROM BLOCK 1 (LAYER 0)
# Then stitch them into a single 8×8 mosaic image
# ============================================================

layer_idx = 0      # First Conv2d layer (64 filters of size 3×3×3)
all_images = []    # Will hold 64 deprocessed numpy images
img_size = 128     # Each filter pattern will be 128×128 pixels

# --- Generate a pattern for each of the 64 filters ---
for filter_index in range(64):
    print(f"\rProcessing filter {filter_index + 1}/64", end="")
    pattern = generate_filter_pattern(
        model, layer_idx, filter_index,
        img_size=img_size, iterations=30, lr=1.0
    )
    img = deprocess_image(pattern)  # Convert tensor → numpy uint8 image
    all_images.append(img)

print("\n✅ All 64 filters generated!")

# --- Stitch all 64 images into a single 8×8 grid ---
margin = 4          # Pixel gap between thumbnails (white border)
n = 8               # 8 images per row and column (8×8 = 64 total)
box_size = all_images[0].shape[0]  # Thumbnail size (128)

# Calculate total canvas size including margins between thumbnails
# Formula: n * box_size + (n - 1) * margin  (n-1 gaps between n items)
full_size = n * box_size + (n - 1) * margin

# Initialize canvas with white pixels (255 = white for uint8)
stitched = np.ones((full_size, full_size, 3), dtype=np.uint8) * 255

# Place each thumbnail into the canvas at the correct position
for i in range(n):      # Row index
    for j in range(n):  # Column index
        img = all_images[i * n + j]  # Linear index: row * n_cols + col
        y_start = i * (box_size + margin)  # Top-left y coordinate
        x_start = j * (box_size + margin)  # Top-left x coordinate
        # Copy the thumbnail into the canvas
        stitched[y_start:y_start + box_size, x_start:x_start + box_size, :] = img

# --- Display the mosaic ---
plt.figure(figsize=(12, 12))
plt.title(f"64 Filter Patterns — features[{layer_idx}] (Block 1, Conv1)",
          fontsize=16, fontweight='bold')
plt.imshow(stitched)
plt.axis("off")
plt.savefig(f"filters_grid_layer_{layer_idx}.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Grad-CAM: Visualizing Class Activation Heatmaps

**Grad-CAM** (Gradient-weighted Class Activation Mapping) answers: *"Which parts of the image were most important for predicting class X?"*

### How Grad-CAM Works

1. **Forward pass**: Get the feature maps from the last convolutional layer  
2. **Backward pass**: Compute gradients of the target class score w.r.t. those feature maps  
3. **Global Average Pooling** of the gradients → importance weight per channel  
4. **Weighted combination** of feature maps → raw heatmap  
5. **ReLU** to keep only positive contributions  
6. **Overlay** on original image  

$$\alpha_k = \frac{1}{Z} \sum_i \sum_j \frac{\partial y^c}{\partial A^k_{ij}} \qquad \text{L}_{\text{Grad-CAM}} = \text{ReLU}\left(\sum_k \alpha_k A^k\right)$$

where $A^k$ is the $k$-th feature map and $y^c$ is the score for class $c$.

In [ ]:
# ============================================================
# LOAD IMAGENET CLASS LABELS
# ImageNet has 1000 classes — we need the label names for display
# ============================================================

# Download a JSON file mapping class index (0-999) → human-readable label
IMAGENET_LABELS_URL = "https://raw.githubusercontent.com/anishathalye/imagenet-simple-labels/master/imagenet-simple-labels.json"
import json
response = requests.get(IMAGENET_LABELS_URL)       # HTTP GET request
imagenet_labels = json.loads(response.text)         # Parse JSON string → Python list

print(f"✅ Loaded {len(imagenet_labels)} ImageNet class labels")  # Should be 1000
print(f"   Examples: {imagenet_labels[:5]}...")  # First 5 classes: tench, goldfish, ...

In [ ]:
# ============================================================
# DOWNLOAD, PREPROCESS, AND CLASSIFY THE ELEPHANT IMAGE
# ============================================================

# Download the elephant image (same as the book uses for Grad-CAM demo)
elephant_url = "https://img-datasets.s3.amazonaws.com/elephant.jpg"
elephant_img = download_image(elephant_url, save_path="elephant.jpg")

# Preprocess: resize → tensor → normalize (same pipeline as the cat image)
elephant_tensor = imagenet_preprocess(elephant_img).unsqueeze(0).to(device)

# --- Run inference (classification) ---
model.eval()  # Ensure model is in evaluation mode
with torch.no_grad():  # Disable gradient computation for faster inference
    logits = model(elephant_tensor)  # Raw class scores (shape: [1, 1000])
    # logits are unnormalized scores — they can be any real number
    
    # Apply softmax to convert logits → probabilities that sum to 1
    # F.softmax(x, dim=1) applies softmax across the class dimension
    # Formula: prob_i = exp(logit_i) / sum(exp(logit_j)) for all j
    probs = F.softmax(logits, dim=1)  # Shape: [1, 1000], each value in [0, 1]

# --- Get top-5 predictions ---
# .topk(5) returns the 5 largest values and their indices
# Returns a named tuple: (values, indices)
top5_probs, top5_indices = probs.topk(5)
top5_probs = top5_probs[0].cpu().numpy()      # Remove batch dim, move to CPU, convert to numpy
top5_indices = top5_indices[0].cpu().numpy()   # Same for indices

# ============================================================
# VISUALIZE: Image + Top-5 Predictions Bar Chart
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LEFT: Show the input image
axes[0].imshow(elephant_img)
axes[0].set_title("Input Image", fontsize=14, fontweight='bold')
axes[0].axis("off")

# RIGHT: Horizontal bar chart of top-5 predictions
labels = [imagenet_labels[i] for i in top5_indices]  # Map indices → class names
# Green for the top prediction, blue for the rest
colors = ['#4CAF50' if i == 0 else '#2196F3' for i in range(5)]

# range(4, -1, -1) = [4, 3, 2, 1, 0] — puts #1 prediction at the top of the chart
bars = axes[1].barh(range(4, -1, -1), top5_probs, color=colors, edgecolor='white')
axes[1].set_yticks(range(4, -1, -1))   # Match tick positions to bar positions
axes[1].set_yticklabels(labels, fontsize=12)  # Label each bar with class name
axes[1].set_xlabel("Probability", fontsize=12)
axes[1].set_title("Top-5 Predictions (VGG16)", fontsize=14, fontweight='bold')

# Annotate each bar with its exact probability value
for i, (bar, prob) in enumerate(zip(bars, top5_probs)):
    axes[1].text(
        bar.get_width() + 0.01,                    # x position: just right of the bar
        bar.get_y() + bar.get_height() / 2,        # y position: vertically centered on the bar
        f"{prob:.4f}",                              # 4 decimal places
        va='center', fontsize=11
    )
axes[1].set_xlim(0, top5_probs[0] * 1.3)  # Add 30% padding to the right

plt.tight_layout()
plt.show()

# Store the top predicted class index for Grad-CAM
top_class = top5_indices[0]
print(f"\n🐘 Top prediction: '{imagenet_labels[top_class]}' (class {top_class}) "
      f"with probability {top5_probs[0]:.4f}")

### 5.1 Implementing Grad-CAM in PyTorch

We use **forward hooks** to capture the last conv layer's output and **backward hooks** to capture gradients — this is the idiomatic PyTorch approach.

In [ ]:
# ============================================================
# GRAD-CAM IMPLEMENTATION
# ============================================================
#
# Grad-CAM uses TWO types of hooks:
#   1. Forward hook:  captures the feature maps (activations) from the target layer
#   2. Backward hook: captures the gradients flowing back through the target layer
#
# The combination of activations + gradients tells us which feature maps
# were most important for a specific class prediction.
# ============================================================

class GradCAM:
    """
    Grad-CAM (Gradient-weighted Class Activation Mapping) implementation.
    
    This class attaches hooks to a target convolutional layer to capture:
      - Forward activations (feature maps): what each filter detected
      - Backward gradients: how important each filter was for the predicted class
    
    The heatmap is computed as a weighted sum of feature maps, where weights
    are the global-average-pooled gradients.
    
    Usage:
        gc = GradCAM(model, model.features[28])  # Target the last conv layer
        heatmap, cls = gc.generate(input_tensor)  # Generate heatmap
        gc.remove_hooks()                         # Clean up
    
    Args:
        model (nn.Module): The full model (e.g., VGG16).
        target_layer (nn.Module): The specific convolutional layer to analyze.
    """

    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None     # Will store gradients from backward hook
        self.activations = None   # Will store activations from forward hook

        # --- Register hooks on the target layer ---
        # Forward hook: called during forward pass, captures the layer's output
        self._forward_hook = target_layer.register_forward_hook(self._save_activation)
        # Backward hook: called during backward pass, captures gradients flowing through this layer
        # register_full_backward_hook is the modern API (replaces deprecated register_backward_hook)
        self._backward_hook = target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        """
        Forward hook callback — stores the layer's output tensor.
        
        Called automatically during model(input). The output contains
        the feature maps that we'll weight by gradient importance.
        """
        # .detach() disconnects from the computation graph to save memory
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        """
        Backward hook callback — stores the gradients of the loss
        w.r.t. this layer's output.
        
        Called automatically during loss.backward(). grad_output[0]
        is the gradient of the loss with respect to this layer's output tensor.
        These gradients tell us how much each spatial location in each
        feature map contributed to the target class score.
        """
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        """
        Generate a Grad-CAM heatmap for a specific class.
        
        Args:
            input_tensor (torch.Tensor): Preprocessed input image, shape (1, 3, 224, 224).
            target_class (int, optional): The class index to generate the heatmap for.
                If None, uses the predicted (argmax) class.
        
        Returns:
            tuple: (heatmap, target_class)
                - heatmap (np.ndarray): 2D array of shape (H', W'), values in [0, 1]
                - target_class (int): The class index used
        """
        self.model.eval()

        # --- Step 1: Forward pass ---
        # This triggers the forward hook, which stores self.activations
        output = self.model(input_tensor)  # Shape: (1, 1000)

        # If no target class specified, use the predicted class
        if target_class is None:
            target_class = output.argmax(dim=1).item()  # .item() converts 0-d tensor → Python int

        # --- Step 2: Zero all existing gradients ---
        # Prevents gradient accumulation from previous calls
        self.model.zero_grad()

        # --- Step 3: Backward pass for the target class ---
        # We want gradients of the target class score (not loss)
        # w.r.t. the feature maps of the target layer
        target_score = output[0, target_class]  # Scalar: the logit for the target class
        # .backward() computes ∂target_score/∂(every tensor with requires_grad=True)
        # retain_graph=True keeps the computation graph so we can call backward again if needed
        target_score.backward(retain_graph=True)

        # --- Step 4: Compute Grad-CAM ---
        # Global Average Pooling of gradients over spatial dimensions (H, W)
        # This gives us one importance weight per channel (α_k in the paper)
        # self.gradients shape: (1, C, H', W') → mean over dims [0, 2, 3] → (C,)
        pooled_gradients = self.gradients.mean(dim=[0, 2, 3])  # Shape: (C,)

        # Get the activations (feature maps) from the forward pass
        activations = self.activations[0]  # Remove batch dim: (1, C, H', W') → (C, H', W')

        # Weight each feature map by its importance score
        # This is equivalent to: weighted_maps = activations * pooled_gradients[:, None, None]
        for i in range(activations.shape[0]):   # Loop over channels
            activations[i] *= pooled_gradients[i]  # Scale the entire feature map by its weight

        # Average the weighted feature maps across the channel dimension
        # This collapses (C, H', W') → (H', W') — a single 2D heatmap
        heatmap = activations.mean(dim=0).cpu().numpy()  # Shape: (H', W')

        # Apply ReLU: keep only positive contributions
        # Negative values represent features that reduce the class score — we ignore them
        heatmap = np.maximum(heatmap, 0)

        # Normalize to [0, 1] for visualization
        heatmap /= (heatmap.max() + 1e-8)  # 1e-8 prevents division by zero

        return heatmap, target_class

    def remove_hooks(self):
        """
        Remove the registered hooks to prevent memory leaks.
        Always call this when done with the GradCAM instance.
        """
        self._forward_hook.remove()
        self._backward_hook.remove()


print("✅ GradCAM class defined.")

In [ ]:
# ============================================================
# APPLY GRAD-CAM TO THE ELEPHANT IMAGE
# ============================================================

# --- Reload the model to clear any stale hooks from previous cells ---
# This is a safety measure: leftover hooks from earlier experiments
# could interfere with Grad-CAM's hooks.
model = models.vgg16(weights=VGG16_Weights.IMAGENET1K_V1).to(device)
model.eval()

# IMPORTANT: Disable in-place ReLU operations
# In-place ReLU (inplace=True) modifies tensors directly, which can corrupt
# the computation graph and cause incorrect gradients during backward().
# We need clean gradients for Grad-CAM, so we switch to out-of-place ReLU.
for m in model.modules():
    if isinstance(m, torch.nn.ReLU):
        m.inplace = False

# --- Target: the last Conv2d layer in VGG16 ---
# features[28] is the last Conv2d before the final MaxPool2d (features[30])
# This is the standard choice for Grad-CAM because it has:
#   - The richest semantic information (deepest layer)
#   - Reasonable spatial resolution (14×14 for 224×224 input)
last_conv_layer = model.features[28]

# --- Generate the Grad-CAM heatmap ---
grad_cam = GradCAM(model, last_conv_layer)
heatmap, predicted_class = grad_cam.generate(elephant_tensor)
grad_cam.remove_hooks()  # Always clean up hooks!

print(f"Predicted class: {predicted_class} ({imagenet_labels[predicted_class]})")
print(f"Heatmap shape: {heatmap.shape}")  # Should be (14, 14) — the spatial size of features[28]

# ============================================================
# VISUALIZE: Original | Raw Heatmap | Overlay
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: Original image
axes[0].imshow(elephant_img)
axes[0].set_title("Original Image", fontsize=14, fontweight='bold')
axes[0].axis("off")

# Panel 2: Raw Grad-CAM heatmap (14×14, shown with 'jet' colormap)
# .matshow() is like imshow but designed for matrices — good for small 2D arrays
axes[1].matshow(heatmap, cmap='jet')  # Jet colormap: blue=cold/low → red=hot/high
axes[1].set_title(f"Grad-CAM Heatmap\n(class: {imagenet_labels[predicted_class]})",
                  fontsize=14, fontweight='bold')
axes[1].axis("off")

# Panel 3: Heatmap overlaid on original image
# Step 3a: Scale heatmap to [0, 255] uint8
heatmap_resized = np.uint8(255 * heatmap)

# Step 3b: Resize heatmap from 14×14 to match the original image dimensions
# using bilinear interpolation for smooth upscaling
heatmap_pil = Image.fromarray(heatmap_resized).resize(
    (elephant_img.width, elephant_img.height), Image.BILINEAR
)
heatmap_np = np.array(heatmap_pil) / 255.0  # Back to [0, 1] for colormap input

# Step 3c: Apply the 'jet' colormap to turn the 1-channel heatmap into RGB
jet_cmap = cm.get_cmap("jet")                          # Get the colormap function
jet_heatmap = jet_cmap(heatmap_np)[:, :, :3]           # Apply it; drop alpha channel (4th channel)
jet_heatmap = (jet_heatmap * 255).astype(np.uint8)     # Scale to [0, 255] uint8

# Step 3d: Superimpose heatmap on original image using weighted blending
# 40% heatmap + 60% original → semi-transparent overlay
elephant_np = np.array(elephant_img)
superimposed = (jet_heatmap * 0.4 + elephant_np * 0.6).astype(np.uint8)

axes[2].imshow(superimposed)
axes[2].set_title("Grad-CAM Overlay", fontsize=14, fontweight='bold')
axes[2].axis("off")

plt.suptitle("Grad-CAM: Where is the network looking?", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("gradcam_elephant.png", dpi=150, bbox_inches='tight')
plt.show()

### 5.2 Grad-CAM for Different Classes (Added Visualization)

A powerful feature of Grad-CAM: we can generate heatmaps for **any class**, not just the predicted one. This reveals what regions the network associates with different concepts.

In [ ]:
# ============================================================
# GRAD-CAM OVERLAY HELPER FUNCTION
# Encapsulates the full pipeline: hooks → heatmap → resize → overlay
# ============================================================

def make_gradcam_overlay(model, target_layer, input_tensor, original_img, target_class):
    """
    Generate a Grad-CAM heatmap and overlay it on the original image.
    
    This function handles the complete pipeline:
      1. Create a GradCAM instance with hooks
      2. Generate the raw heatmap
      3. Remove hooks (cleanup)
      4. Resize heatmap to match the original image
      5. Apply colormap and blend with original image
    
    Args:
        model (nn.Module):        The classification model.
        target_layer (nn.Module): The convolutional layer to target.
        input_tensor (Tensor):    Preprocessed input image [1, 3, 224, 224].
        original_img (PIL.Image): Original image for overlay (can be any size).
        target_class (int or None): Class index to visualize. None = use predicted class.
    
    Returns:
        tuple: (heatmap, overlay)
            - heatmap (np.ndarray):  Raw normalized heatmap [0, 1], shape (H', W')
            - overlay (np.ndarray):  Blended image, shape (H, W, 3), uint8
    """
    # Create GradCAM, generate heatmap, immediately clean up hooks
    gc = GradCAM(model, target_layer)
    heatmap, _ = gc.generate(input_tensor, target_class=target_class)
    gc.remove_hooks()

    # Resize the small heatmap (e.g., 14×14) to the original image's dimensions
    heatmap_resized = np.uint8(255 * heatmap)
    heatmap_pil = Image.fromarray(heatmap_resized).resize(
        (original_img.width, original_img.height), Image.BILINEAR  # Smooth upscaling
    )
    heatmap_np = np.array(heatmap_pil) / 255.0  # Normalize back to [0, 1]

    # Apply jet colormap: grayscale [0,1] → RGB color
    jet_cmap = cm.get_cmap("jet")
    jet_heatmap = (jet_cmap(heatmap_np)[:, :, :3] * 255).astype(np.uint8)  # Drop alpha, scale

    # Blend: 40% heatmap color + 60% original image
    original_np = np.array(original_img)
    superimposed = (jet_heatmap * 0.4 + original_np * 0.6).astype(np.uint8)

    return heatmap, superimposed


# ============================================================
# COMPARE GRAD-CAM HEATMAPS FOR THE TOP-5 PREDICTED CLASSES
# ============================================================
# This shows that Grad-CAM is class-discriminative:
# different classes highlight different image regions.

# Re-classify the elephant image to get fresh top-5 predictions
with torch.no_grad():
    logits = model(elephant_tensor)
    probs = F.softmax(logits, dim=1)
_, top5_idx = probs.topk(5)
top5_idx = top5_idx[0].cpu().numpy()

# Create a 1×5 grid showing Grad-CAM for each of the top-5 classes
fig, axes = plt.subplots(1, 5, figsize=(25, 5))
fig.suptitle("Grad-CAM Heatmaps for Top-5 Predicted Classes",
             fontsize=16, fontweight='bold', y=1.05)

for i, cls_idx in enumerate(top5_idx):
    # Generate heatmap + overlay for this specific class
    heatmap, overlay = make_gradcam_overlay(
        model, model.features[28], elephant_tensor, elephant_img, cls_idx
    )
    axes[i].imshow(overlay)
    prob = probs[0, cls_idx].item()  # Get the probability for this class
    axes[i].set_title(f"#{i+1}: {imagenet_labels[cls_idx]}\n(p={prob:.4f})",
                      fontsize=11, fontweight='bold')
    axes[i].axis("off")

plt.tight_layout()
plt.show()

### 5.3 Grad-CAM at Different Layers (Added Visualization)

Grad-CAM doesn't have to target only the last conv layer. Let's see how the heatmap changes when we use **different convolutional layers** — earlier layers produce more spatially precise but less semantically meaningful heatmaps.

In [ ]:
# ============================================================
# GRAD-CAM FROM DIFFERENT CONVOLUTIONAL LAYERS
# ============================================================
# Deeper layers → coarser spatial resolution but richer semantic content
# Shallower layers → finer spatial detail but less class-specific

# Map descriptive names → actual layer modules
# We pick the last Conv2d from each block
target_layers = {
    "Block 1 (features.2)":  model.features[2],   # Output: 224×224, 64 channels
    "Block 2 (features.7)":  model.features[7],   # Output: 112×112, 128 channels
    "Block 3 (features.14)": model.features[14],  # Output: 56×56, 256 channels
    "Block 4 (features.21)": model.features[21],  # Output: 28×28, 512 channels
    "Block 5 (features.28)": model.features[28],  # Output: 14×14, 512 channels
}

# Create a 1×6 grid: original + 5 heatmaps from different depths
fig, axes = plt.subplots(1, len(target_layers) + 1, figsize=(24, 4))
fig.suptitle("Grad-CAM at Different Network Depths", fontsize=16, fontweight='bold', y=1.08)

# First panel: the original image for reference
axes[0].imshow(elephant_img)
axes[0].set_title("Original", fontsize=12, fontweight='bold')
axes[0].axis("off")

# Remaining panels: Grad-CAM overlays from each block
for i, (name, layer) in enumerate(target_layers.items()):
    heatmap, overlay = make_gradcam_overlay(
        model, layer, elephant_tensor, elephant_img, target_class=top_class
    )
    axes[i + 1].imshow(overlay)
    axes[i + 1].set_title(name, fontsize=10, fontweight='bold')
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()

print("📊 Observation: Deeper layers → coarser but more semantically meaningful heatmaps")
print("   Shallow layers → spatially precise but noisy, highlight low-level features")

### 5.4 Grad-CAM on the Cat Image

Let's also apply Grad-CAM to our cat image, and compare with the cat's intermediate activations from Section 3.

In [ ]:
# ============================================================
# GRAD-CAM ANALYSIS FOR THE CAT IMAGE
# ============================================================

# Preprocess the cat image (same pipeline as before)
cat_tensor = imagenet_preprocess(cat_img).unsqueeze(0).to(device)

# --- Classify the cat image ---
with torch.no_grad():
    cat_logits = model(cat_tensor)               # Raw logits: (1, 1000)
    cat_probs = F.softmax(cat_logits, dim=1)     # Probabilities: (1, 1000)
    # Get top-5 predictions: returns (values_tensor, indices_tensor)
    cat_top5_probs, cat_top5_idx = cat_probs.topk(5)

# Convert to numpy for easy indexing
cat_top5_probs = cat_top5_probs[0].cpu().numpy()
cat_top5_idx = cat_top5_idx[0].cpu().numpy()

# ============================================================
# VISUALIZE: 2×3 grid
#   Top row:    Original | Raw heatmap | Overlay (top class)
#   Bottom row: Top-3 class overlays
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Grad-CAM Analysis — Cat Image", fontsize=16, fontweight='bold', y=1.02)

# --- Top row ---

# Panel (0,0): Original image
axes[0, 0].imshow(cat_img)
axes[0, 0].set_title("Original", fontsize=13, fontweight='bold')
axes[0, 0].axis("off")

# Panel (0,1): Raw Grad-CAM heatmap for the top predicted class
cat_class = cat_top5_idx[0]  # Index of the most likely class
heatmap, overlay = make_gradcam_overlay(
    model, model.features[28], cat_tensor, cat_img, cat_class
)
axes[0, 1].matshow(heatmap, cmap='jet')  # Display the raw 14×14 heatmap
axes[0, 1].set_title(f"Heatmap: {imagenet_labels[cat_class]}", fontsize=13, fontweight='bold')
axes[0, 1].axis("off")

# Panel (0,2): Heatmap overlaid on original image
axes[0, 2].imshow(overlay)
axes[0, 2].set_title(f"Overlay: {imagenet_labels[cat_class]}", fontsize=13, fontweight='bold')
axes[0, 2].axis("off")

# --- Bottom row: Grad-CAM overlays for top-3 predicted classes ---
for i in range(3):
    cls = cat_top5_idx[i]  # Class index for the i-th prediction
    _, overlay = make_gradcam_overlay(
        model, model.features[28], cat_tensor, cat_img, cls
    )
    axes[1, i].imshow(overlay)
    axes[1, i].set_title(
        f"#{i+1}: {imagenet_labels[cls]} (p={cat_top5_probs[i]:.3f})",
        fontsize=11, fontweight='bold'
    )
    axes[1, i].axis("off")

plt.tight_layout()
plt.show()

---
## 6. Comparing Architectures: VGG16 vs ResNet50 (Added Section)

Do different architectures "look" at the same regions? Let's compare Grad-CAM from **VGG16** and **ResNet50** on the same images.

In [ ]:
# ============================================================
# COMPARE GRAD-CAM: VGG16 vs ResNet50
# ============================================================
# Different architectures may focus on different image regions.
# ResNet50 uses residual connections (skip connections) which
# allow gradients to flow more directly, often producing
# sharper, more localized heatmaps.

# --- Load ResNet50 with pretrained ImageNet weights ---
resnet = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1).to(device)
resnet.eval()

# --- Identify the last convolutional layer in ResNet50 ---
# ResNet50 architecture: layer1 → layer2 → layer3 → layer4
# Each "layer" is a sequence of Bottleneck blocks.
# layer4[-1] = last Bottleneck block; .conv3 = its final 1×1 Conv2d
# This is analogous to features[28] in VGG16.
resnet_last_conv = resnet.layer4[-1].conv3

# --- Prepare test images ---
test_images = [
    ("Cat", cat_img, cat_tensor),          # Tuple: (name, PIL image, preprocessed tensor)
    ("Elephant", elephant_img, elephant_tensor),
]

# --- Create a 2×3 comparison grid ---
# Rows = different images, Columns = Original | VGG16 Grad-CAM | ResNet50 Grad-CAM
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Grad-CAM Comparison: VGG16 vs ResNet50", fontsize=18, fontweight='bold', y=1.02)

for row, (name, pil_img, tensor) in enumerate(test_images):
    # Column 0: Original image
    axes[row, 0].imshow(pil_img)
    axes[row, 0].set_title(f"{name} — Original", fontsize=13, fontweight='bold')
    axes[row, 0].axis("off")

    # Column 1: VGG16 Grad-CAM overlay
    # target_class=None → auto-detect the predicted class
    _, vgg_overlay = make_gradcam_overlay(
        model, model.features[28], tensor, pil_img, target_class=None
    )
    axes[row, 1].imshow(vgg_overlay)
    axes[row, 1].set_title(f"{name} — VGG16", fontsize=13, fontweight='bold')
    axes[row, 1].axis("off")

    # Column 2: ResNet50 Grad-CAM overlay
    _, resnet_overlay = make_gradcam_overlay(
        resnet, resnet_last_conv, tensor, pil_img, target_class=None
    )
    axes[row, 2].imshow(resnet_overlay)
    axes[row, 2].set_title(f"{name} — ResNet50", fontsize=13, fontweight='bold')
    axes[row, 2].axis("off")

plt.tight_layout()
plt.savefig("gradcam_vgg_vs_resnet.png", dpi=150, bbox_inches='tight')
plt.show()

print("📊 Both networks focus on similar regions, but ResNet's skip connections")
print("   often produce more precise, well-localized heatmaps.")

---
## 7. Guided Backpropagation (Added Advanced Technique)

**Guided Backpropagation** modifies the backward pass so that only **positive gradients** flow through ReLU layers. This produces high-resolution, pixel-level visualizations of what the network detects.

When combined with Grad-CAM → **Guided Grad-CAM** = element-wise product that is both high-resolution AND class-discriminative.

In [ ]:
# ============================================================
# GUIDED BACKPROPAGATION IMPLEMENTATION
# ============================================================
#
# STANDARD BACKPROP through ReLU:
#   Forward: out = max(0, in)        → passes positive values, zeros negatives
#   Backward: grad_in = grad_out * (in > 0)  → gates gradients by forward-pass sign
#
# GUIDED BACKPROP through ReLU:
#   Backward: grad_in = grad_out * (in > 0) * (grad_out > 0)
#   The extra (grad_out > 0) mask blocks NEGATIVE gradients from flowing backward.
#   This means we only visualize features that POSITIVELY contribute to the output.
#
# The result is a cleaner, more interpretable visualization at pixel resolution.
# ============================================================

class GuidedBackprop:
    """
    Guided Backpropagation: modifies ReLU backward pass to only
    propagate positive gradients in both the forward and backward directions.
    
    This produces high-resolution, pixel-level attribution maps that
    show which pixels contribute positively to the target class.
    
    Args:
        model (nn.Module): The model to analyze.
    """

    def __init__(self, model):
        self.model = model
        self.hooks = []           # Store hook handles for cleanup
        self._register_hooks()    # Attach guided ReLU hooks to all ReLU layers

    def _register_hooks(self):
        """
        Replace the backward pass of every ReLU in the model with
        a 'guided' version that only passes positive gradients.
        """
        def guided_relu_hook(module, grad_in, grad_out):
            """
            Backward hook for guided backpropagation.
            
            Clamps the incoming gradient to be non-negative.
            This is the "guided" part: we only propagate gradients
            that correspond to positive contributions.
            
            Args:
                module: The ReLU module (not used but required by hook signature).
                grad_in:  Tuple of gradients w.r.t. the layer's inputs.
                grad_out: Tuple of gradients w.r.t. the layer's outputs.
            
            Returns:
                Tuple: Modified gradient (only positive values).
            """
            # torch.clamp(x, min=0) is equivalent to ReLU on the gradient
            return (torch.clamp(grad_in[0], min=0.0),)

        # Attach this hook to every ReLU module in the model
        for module in self.model.modules():  # .modules() iterates ALL modules recursively
            if isinstance(module, nn.ReLU):
                h = module.register_backward_hook(guided_relu_hook)
                self.hooks.append(h)

    def generate(self, input_tensor, target_class=None):
        """
        Compute guided backpropagation saliency map.
        
        Args:
            input_tensor (Tensor): Preprocessed image [1, 3, 224, 224].
            target_class (int, optional): Class to compute attribution for.
        
        Returns:
            np.ndarray: Gradient image of shape (224, 224, 3) — the saliency map.
        """
        # Clone the input and enable gradient tracking
        # We need gradients w.r.t. the input image pixels
        input_tensor = input_tensor.clone().detach().requires_grad_(True)

        # Forward pass
        output = self.model(input_tensor)  # (1, 1000)

        if target_class is None:
            target_class = output.argmax(dim=1).item()

        # Backward pass for the target class
        self.model.zero_grad()
        target_score = output[0, target_class]
        target_score.backward()

        # The gradient of the input IS the guided backprop result
        # input_tensor.grad has shape (1, 3, 224, 224)
        guided_grads = input_tensor.grad[0].cpu().numpy()   # (3, 224, 224)
        guided_grads = np.transpose(guided_grads, (1, 2, 0))  # → (224, 224, 3) for display
        return guided_grads

    def remove_hooks(self):
        """Remove all registered hooks to restore normal backprop behavior."""
        for h in self.hooks:
            h.remove()


# ============================================================
# APPLY GUIDED BACKPROP + COMBINE WITH GRAD-CAM
# ============================================================

# --- Guided Backpropagation ---
gbp = GuidedBackprop(model)
guided_grads_elephant = gbp.generate(elephant_tensor)  # Shape: (224, 224, 3)
guided_grads_cat = gbp.generate(cat_tensor)            # Shape: (224, 224, 3)
gbp.remove_hooks()  # Clean up guided ReLU hooks

# --- Grad-CAM heatmaps (for combination into Guided Grad-CAM) ---
gc = GradCAM(model, model.features[28])
elephant_heatmap, _ = gc.generate(elephant_tensor)
gc.remove_hooks()

gc2 = GradCAM(model, model.features[28])
cat_heatmap, _ = gc2.generate(cat_tensor)
gc2.remove_hooks()


def process_guided_grads(grads):
    """
    Normalize guided backprop gradients for display.
    
    Shifts the gradient values to [0, 1] range using min-max normalization.
    
    Args:
        grads (np.ndarray): Raw gradient array.
    
    Returns:
        np.ndarray: Normalized array in [0, 1].
    """
    grads = grads - grads.min()        # Shift minimum to 0
    grads /= (grads.max() + 1e-8)      # Scale maximum to 1
    return grads


# ============================================================
# VISUALIZE: Original | Guided Backprop | Grad-CAM | Guided Grad-CAM
# ============================================================
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
fig.suptitle("Guided Backpropagation & Guided Grad-CAM", fontsize=18, fontweight='bold', y=1.02)

for row, (name, pil_img, grads, heatmap) in enumerate([
    ("Elephant", elephant_img, guided_grads_elephant, elephant_heatmap),
    ("Cat", cat_img, guided_grads_cat, cat_heatmap)
]):
    # Column 0: Original image
    axes[row, 0].imshow(pil_img)
    axes[row, 0].set_title(f"{name} — Original", fontsize=13, fontweight='bold')
    axes[row, 0].axis("off")

    # Column 1: Guided Backprop — pixel-level attribution
    # Shows fine-grained details but is NOT class-discriminative
    gbp_display = process_guided_grads(grads)
    axes[row, 1].imshow(gbp_display)
    axes[row, 1].set_title(f"{name} — Guided Backprop", fontsize=13, fontweight='bold')
    axes[row, 1].axis("off")

    # Column 2: Grad-CAM — class-discriminative but low resolution
    # Resize the 14×14 heatmap to 224×224 for display
    heatmap_resized = np.array(Image.fromarray(
        np.uint8(255 * heatmap)
    ).resize((224, 224), Image.BILINEAR)) / 255.0
    axes[row, 2].imshow(heatmap_resized, cmap='jet')
    axes[row, 2].set_title(f"{name} — Grad-CAM", fontsize=13, fontweight='bold')
    axes[row, 2].axis("off")

    # Column 3: Guided Grad-CAM = element-wise product of Guided Backprop × Grad-CAM
    # This combines:
    #   - High-resolution detail from Guided Backprop
    #   - Class-discriminative localization from Grad-CAM
    # heatmap_resized[:, :, np.newaxis] adds a channel dim: (224,224) → (224,224,1)
    # so it broadcasts across the 3 RGB channels of gbp_display
    guided_gradcam = gbp_display * heatmap_resized[:, :, np.newaxis]
    guided_gradcam = process_guided_grads(guided_gradcam)  # Re-normalize for display
    axes[row, 3].imshow(guided_gradcam)
    axes[row, 3].set_title(f"{name} — Guided Grad-CAM", fontsize=13, fontweight='bold')
    axes[row, 3].axis("off")

plt.tight_layout()
plt.savefig("guided_gradcam.png", dpi=150, bbox_inches='tight')
plt.show()

print("📊 Guided Grad-CAM combines the best of both:")
print("   • High-resolution detail from Guided Backprop")
print("   • Class-discriminative localization from Grad-CAM")

---
## 8. Visualizing the Latent Space with t-SNE & PCA (Added Section)

The book mentions visualizing the latent space of a ConvNet. Let's take this further by:
1. Extracting **feature embeddings** from VGG16's penultimate FC layer for multiple images
2. Visualizing them with **PCA** and **t-SNE** to see how the network organizes different classes

We'll use a small set of ImageNet-style images across several categories.

In [ ]:
# ============================================================
# EXTRACT FEATURE EMBEDDINGS FROM VGG16's PENULTIMATE FC LAYER
# ============================================================
#
# The penultimate FC layer (classifier[5]) outputs a 4096-dimensional vector.
# This vector is the network's "internal representation" of the image —
# a rich, high-level feature that captures semantic content.
#
# By extracting these embeddings for many images and projecting them to 2D,
# we can visualize how the network organizes different classes in its
# learned representation space.
# ============================================================

# --- Fix: Clear stale cached scipy/sklearn modules from kernel memory ---
# scipy was upgraded from 1.13.1 → 1.17.0 but the kernel still has old
# compiled .so files in its module cache. We must purge them and
# invalidate Python's import file-finder caches before re-importing.
import sys as _sys
import importlib as _importlib
_mods_to_remove = [k for k in _sys.modules if k.startswith('scipy') or k.startswith('sklearn')]
for _m in _mods_to_remove:
    del _sys.modules[_m]
_importlib.invalidate_caches()          # force Python to re-scan directories
print(f"Cleared {len(_mods_to_remove)} cached modules & invalidated import caches")

from sklearn.manifold import TSNE    # t-SNE: non-linear dimensionality reduction
from sklearn.decomposition import PCA  # PCA: linear dimensionality reduction


class EmbeddingExtractor(nn.Module):
    """
    Extract the penultimate FC layer's output (4096-d embedding) from VGG16.
    
    VGG16's classifier is:
        classifier[0]: Linear(25088, 4096)  ← FC1
        classifier[1]: ReLU
        classifier[2]: Dropout
        classifier[3]: Linear(4096, 4096)   ← FC2 (penultimate)
        classifier[4]: ReLU
        classifier[5]: Dropout
        classifier[6]: Linear(4096, 1000)   ← Final classification layer
    
    We take everything EXCEPT classifier[6] to get the 4096-d embedding.
    
    Args:
        model: The full VGG16 model.
    """

    def __init__(self, model):
        super().__init__()
        # ---- Build the feature extraction backbone (all conv + pool layers) ----
        self.features = model.features        # Sequential of conv/relu/pool layers
        self.avgpool = model.avgpool           # Adaptive average pool → 7×7
        
        # ---- Build truncated classifier (exclude final Linear layer) ----
        # classifier[0:6] gives us: FC1 → ReLU → Dropout → FC2 → ReLU → Dropout
        self.classifier = nn.Sequential(*list(model.classifier.children())[:-1])

    def forward(self, x):
        # 1. Run through convolutional feature extractor
        x = self.features(x)                  # (B, 512, H', W')
        # 2. Adaptive average pooling → fixed 7×7 spatial size
        x = self.avgpool(x)                   # (B, 512, 7, 7)
        # 3. Flatten to 1D vector: 512 * 7 * 7 = 25088
        x = torch.flatten(x, 1)              # (B, 25088)
        # 4. Pass through truncated classifier → 4096-d embedding
        x = self.classifier(x)               # (B, 4096)
        return x


# ---- Instantiate the embedding extractor and move to device ----
embedding_model = EmbeddingExtractor(model).to(device)  # Reuse the VGG16 model
embedding_model.eval()  # Set to eval mode (disables dropout, uses running BN stats)
print(f"Embedding model ready on {device}")
print(f"Output embedding dimension: 4096")

# ---- Load CIFAR-10 test set to extract embeddings ----
# CIFAR-10 has 10 classes of 32×32 images. We'll resize them to 224×224
# (VGG16's expected input) and use ImageNet normalization.
cifar_transform = transforms.Compose([
    transforms.Resize((224, 224)),            # Upscale 32×32 → 224×224 for VGG16
    transforms.ToTensor(),                     # Convert PIL → tensor, scale to [0,1]
    transforms.Normalize(                      # ImageNet normalization statistics
        mean=[0.485, 0.456, 0.406],           # Subtract ImageNet channel means
        std=[0.229, 0.224, 0.225]             # Divide by ImageNet channel stds
    )
])

# Download CIFAR-10 test split (10,000 images across 10 classes)
cifar_test = torchvision.datasets.CIFAR10(
    root='./data',          # Local cache directory
    train=False,            # Use test split (not training)
    download=True,          # Download if not already cached
    transform=cifar_transform  # Apply our preprocessing pipeline
)

# CIFAR-10 class names for labeling
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                 'dog', 'frog', 'horse', 'ship', 'truck']

# ---- Extract embeddings for a subset of 500 images ----
n_samples = 500  # Number of images to embed (keep manageable for t-SNE)

# Create a DataLoader for batched processing
subset_indices = list(range(n_samples))  # First 500 images
subset = torch.utils.data.Subset(cifar_test, subset_indices)  # Create subset
loader = torch.utils.data.DataLoader(subset, batch_size=32, shuffle=False)

all_embeddings = []  # Will collect (B, 4096) tensors
all_labels = []      # Corresponding class labels

with torch.no_grad():  # Disable gradient computation for efficiency
    for images, labels in loader:
        images = images.to(device)             # Move batch to GPU/MPS
        embeddings = embedding_model(images)   # Forward pass → (B, 4096)
        all_embeddings.append(embeddings.cpu())  # Move back to CPU for numpy
        all_labels.append(labels)

# Concatenate all batches into single tensors
all_embeddings = torch.cat(all_embeddings, dim=0).numpy()  # (500, 4096)
all_labels = torch.cat(all_labels, dim=0).numpy()          # (500,)

print(f"\nExtracted {all_embeddings.shape[0]} embeddings of dimension {all_embeddings.shape[1]}")
print(f"Label distribution: {dict(zip(*np.unique(all_labels, return_counts=True)))}")

In [ ]:
# ============================================================
# VISUALIZE EMBEDDINGS WITH PCA AND t-SNE
# ============================================================
#
# PCA (Principal Component Analysis):
#   - Linear projection that finds directions of maximum variance
#   - Fast, deterministic, preserves global structure
#   - May miss non-linear relationships
#
# t-SNE (t-Distributed Stochastic Neighbor Embedding):
#   - Non-linear method that preserves LOCAL neighborhood structure
#   - Great for revealing clusters and groups
#   - Slow, stochastic (different runs give different results)
#   - Distances between clusters are NOT meaningful
# ============================================================

# Use the correct variable names from Cell 44
embeddings = all_embeddings   # (500, 4096) numpy array
labels = all_labels           # (500,) numpy array

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle("Latent Space Visualization — VGG16 Embeddings (4096-d → 2-d)",
             fontsize=16, fontweight='bold', y=1.02)

# Get 10 distinct colors from the 'tab10' colormap (one per CIFAR class)
colors_map = plt.cm.tab10(np.linspace(0, 1, 10))

# --- LEFT: PCA projection ---
pca = PCA(n_components=2)  # Project 4096-d → 2-d
embeddings_pca = pca.fit_transform(embeddings)  # Fit PCA and transform in one step

# Plot each class with a different color
for class_idx in range(10):
    mask = labels == class_idx  # Boolean mask for this class
    axes[0].scatter(
        embeddings_pca[mask, 0],  # PC1 values
        embeddings_pca[mask, 1],  # PC2 values
        c=[colors_map[class_idx]],  # Color for this class
        label=cifar_classes[class_idx],  # Legend label
        alpha=0.7,        # Semi-transparent dots
        s=50,             # Marker size
        edgecolors='white', linewidth=0.5  # White border on dots
    )

# Display the total variance explained by the first 2 PCs
# This tells us how much information is preserved in the 2D projection
axes[0].set_title(
    f"PCA (explains {pca.explained_variance_ratio_.sum()*100:.1f}% variance)",
    fontsize=14, fontweight='bold'
)
axes[0].legend(loc='best', fontsize=9, ncol=2)
axes[0].set_xlabel("PC1", fontsize=12)  # First principal component
axes[0].set_ylabel("PC2", fontsize=12)  # Second principal component

# --- RIGHT: t-SNE projection ---
tsne = TSNE(
    n_components=2,    # Project to 2D
    random_state=42,   # Fix seed for reproducibility
    perplexity=30,     # Effective number of neighbors (typical: 5-50)
    max_iter=1000      # Optimization iterations (renamed from n_iter in sklearn 1.8+)
)
embeddings_tsne = tsne.fit_transform(embeddings)  # Fit and transform

for class_idx in range(10):
    mask = labels == class_idx
    axes[1].scatter(
        embeddings_tsne[mask, 0], embeddings_tsne[mask, 1],
        c=[colors_map[class_idx]], label=cifar_classes[class_idx],
        alpha=0.7, s=50, edgecolors='white', linewidth=0.5
    )

axes[1].set_title("t-SNE", fontsize=14, fontweight='bold')
axes[1].legend(loc='best', fontsize=9, ncol=2)
axes[1].set_xlabel("t-SNE 1", fontsize=12)
axes[1].set_ylabel("t-SNE 2", fontsize=12)

plt.tight_layout()
plt.savefig("latent_space_pca_tsne.png", dpi=150, bbox_inches='tight')
plt.show()

print("📊 Key Observations:")
print("   • Semantically similar classes (e.g., cat/dog, car/truck) cluster nearby")
print("   • t-SNE reveals non-linear cluster structure better than PCA")
print("   • Well-separated clusters → features that are useful for classification")

### 8.1 How Embeddings Evolve Through Layers (Added Visualization)

Let's see how the t-SNE visualization changes when we use features from **different depths** of the network. This shows the progressive disentangling of class representations.

In [ ]:
# ============================================================
# t-SNE AT DIFFERENT NETWORK DEPTHS
# Shows how class separability improves with depth
# ============================================================
#
# At shallow layers, features are generic (edges, colors) and classes
# are all mixed together in feature space.
# At deeper layers, features become class-specific and clusters emerge.
# This is the essence of "representation learning".

# Define feature extractors at three different depths
depth_configs = {
    # nn.Sequential(*list(...)[:N]) takes the first N layers of the feature extractor
    "After Block 1 (features[:5])":  nn.Sequential(*list(model.features.children())[:5]),
    "After Block 3 (features[:17])": nn.Sequential(*list(model.features.children())[:17]),
    "After Block 5 (features[:31])": nn.Sequential(*list(model.features.children())[:31]),
}

# --- Select a subset of CIFAR images (15 per class = 150 total) ---
subset_indices = []
for class_idx in range(10):
    indices = [i for i, (_, label) in enumerate(cifar_test) if label == class_idx][:15]
    subset_indices.extend(indices)

# --- Extract features and visualize for each depth ---
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
fig.suptitle("t-SNE at Different Network Depths — Feature Disentangling",
             fontsize=16, fontweight='bold', y=1.05)

for plot_idx, (name, feature_extractor) in enumerate(depth_configs.items()):
    feature_extractor = feature_extractor.to(device).eval()

    feats = []  # Feature vectors for this depth
    lbls = []   # Corresponding labels

    for idx in subset_indices:
        img, label = cifar_test[idx]
        img = img.unsqueeze(0).to(device)
        with torch.no_grad():
            feat = feature_extractor(img)  # Output shape varies by depth
            # F.adaptive_avg_pool2d(feat, 1) reduces spatial dims to 1×1
            # This converts (1, C, H, W) → (1, C, 1, 1)
            # .flatten() then gives us a C-dimensional vector
            # This makes features comparable across different spatial resolutions
            feat = F.adaptive_avg_pool2d(feat, 1).flatten()
        feats.append(feat.cpu().numpy())
        lbls.append(label)

    feats = np.array(feats)  # Shape: (150, C) where C varies by depth
    lbls = np.array(lbls)

    # Apply t-SNE to reduce to 2D
    # Note: max_iter replaced n_iter in scikit-learn 1.8+
    tsne = TSNE(n_components=2, random_state=42, perplexity=20, max_iter=1000)
    feats_2d = tsne.fit_transform(feats)

    # Plot with class-colored dots
    for class_idx in range(10):
        mask = lbls == class_idx
        axes[plot_idx].scatter(
            feats_2d[mask, 0], feats_2d[mask, 1],
            c=[colors_map[class_idx]],
            label=cifar_classes[class_idx],
            alpha=0.7, s=40, edgecolors='white', linewidth=0.5
        )

    axes[plot_idx].set_title(name, fontsize=12, fontweight='bold')
    axes[plot_idx].legend(loc='best', fontsize=7, ncol=2)
    print(f"  ✅ {name}")

plt.tight_layout()
plt.savefig("tsne_across_depths.png", dpi=150, bbox_inches='tight')
plt.show()

print("\n📊 Observation: Classes become increasingly well-separated in deeper layers")
print("   → The network progressively builds more discriminative representations")

---
## 9. Occlusion Sensitivity Analysis (Added Technique)

**Occlusion Sensitivity** is another interpretability method: we systematically **occlude** (cover) patches of the image and observe how the prediction confidence changes. Regions where occlusion causes the biggest drop → the most important regions.

This provides a model-agnostic sanity check for Grad-CAM results.

In [ ]:
# ============================================================
# OCCLUSION SENSITIVITY ANALYSIS
# ============================================================
#
# Algorithm:
#   1. Get the baseline probability for the target class (without occlusion)
#   2. Slide a gray patch across the image at regular intervals
#   3. For each position, measure the new probability
#   4. Sensitivity = baseline_prob - occluded_prob
#      (large positive value → this region was important)
#
# This is a "brute force" approach — no gradients needed!
# It's model-agnostic but slow: O(n²) forward passes.
# ============================================================

def occlusion_sensitivity(model, img_tensor, target_class, patch_size=32, stride=8):
    """
    Compute an occlusion sensitivity map by sliding a gray patch across the image.
    
    For each patch position, we measure how much the target class probability
    drops when that region is occluded. Large drops → important regions.
    
    Args:
        model (nn.Module):      The classification model.
        img_tensor (Tensor):    Preprocessed input image [1, 3, H, W].
        target_class (int):     Class index to track sensitivity for.
        patch_size (int):       Size of the occluding square patch (pixels).
        stride (int):           Step size between patch positions (pixels).
            Smaller stride → higher resolution map but more forward passes.
    
    Returns:
        np.ndarray: 2D sensitivity map where each value is the probability drop
            when that patch position is occluded.
    """
    model.eval()
    _, _, H, W = img_tensor.shape  # Get spatial dimensions (224, 224)

    # --- Step 1: Get baseline prediction (no occlusion) ---
    with torch.no_grad():
        baseline_probs = F.softmax(model(img_tensor), dim=1)
        baseline_prob = baseline_probs[0, target_class].item()  # Scalar: baseline confidence

    # --- Step 2: Calculate the grid dimensions ---
    # Number of patch positions = (image_size - patch_size) / stride + 1
    n_rows = (H - patch_size) // stride + 1  # Vertical positions
    n_cols = (W - patch_size) // stride + 1  # Horizontal positions
    sensitivity_map = np.zeros((n_rows, n_cols))  # Initialize empty map

    # --- Step 3: Slide the patch across the image ---
    for i in range(n_rows):
        for j in range(n_cols):
            # Clone the original image (don't modify it!)
            occluded = img_tensor.clone()

            # Calculate patch boundaries
            y_start = i * stride
            x_start = j * stride

            # Set the patch region to 0 (gray in normalized space)
            # This effectively "hides" this region from the network
            occluded[:, :, y_start:y_start+patch_size, x_start:x_start+patch_size] = 0

            # Forward pass with the occluded image
            with torch.no_grad():
                probs = F.softmax(model(occluded), dim=1)
                prob = probs[0, target_class].item()

            # Sensitivity = how much the probability DROPPED
            # Positive values → occluding this region hurt the prediction (important region)
            # Negative values → occluding this region helped (distracting region)
            sensitivity_map[i, j] = baseline_prob - prob

    return sensitivity_map


# ============================================================
# COMPUTE OCCLUSION SENSITIVITY FOR THE ELEPHANT IMAGE
# ============================================================
print("Computing occlusion sensitivity (this may take a minute)...")
occ_map_elephant = occlusion_sensitivity(
    model, elephant_tensor, top_class,
    patch_size=32,  # 32×32 pixel occluding patch
    stride=14       # Move the patch 14 pixels at a time
)

# --- Also compute Grad-CAM for comparison ---
gc = GradCAM(model, model.features[28])
gradcam_heatmap, _ = gc.generate(elephant_tensor, target_class=top_class)
gc.remove_hooks()

# ============================================================
# VISUALIZE: Original | Occlusion Map | Occlusion Overlay | Grad-CAM Overlay
# ============================================================
fig, axes = plt.subplots(1, 4, figsize=(22, 5))
fig.suptitle("Occlusion Sensitivity vs Grad-CAM Comparison",
             fontsize=16, fontweight='bold', y=1.05)

# Panel 1: Original image
axes[0].imshow(elephant_img)
axes[0].set_title("Original", fontsize=13, fontweight='bold')
axes[0].axis("off")

# Panel 2: Raw occlusion sensitivity map
# 'hot' colormap: black → red → yellow → white (dark=low, bright=high)
# interpolation='bilinear' smooths the low-resolution map
axes[1].imshow(occ_map_elephant, cmap='hot', interpolation='bilinear')
axes[1].set_title("Occlusion Sensitivity\n(bright = important)", fontsize=13, fontweight='bold')
axes[1].axis("off")

# Panel 3: Occlusion map overlaid on original image
# Normalize the occlusion map to [0, 255] and resize to match original image
occ_resized = np.array(Image.fromarray(
    np.uint8(255 * (occ_map_elephant - occ_map_elephant.min()) /
             (occ_map_elephant.max() - occ_map_elephant.min() + 1e-8))
).resize((elephant_img.width, elephant_img.height), Image.BILINEAR)) / 255.0

# Apply jet colormap and blend with original
jet_occ = (cm.get_cmap("jet")(occ_resized)[:, :, :3] * 255).astype(np.uint8)
occ_overlay = (jet_occ * 0.4 + np.array(elephant_img) * 0.6).astype(np.uint8)
axes[2].imshow(occ_overlay)
axes[2].set_title("Occlusion Overlay", fontsize=13, fontweight='bold')
axes[2].axis("off")

# Panel 4: Grad-CAM overlay for side-by-side comparison
_, gradcam_overlay = make_gradcam_overlay(
    model, model.features[28], elephant_tensor, elephant_img, top_class
)
axes[3].imshow(gradcam_overlay)
axes[3].set_title("Grad-CAM Overlay", fontsize=13, fontweight='bold')
axes[3].axis("off")

plt.tight_layout()
plt.savefig("occlusion_vs_gradcam.png", dpi=150, bbox_inches='tight')
plt.show()

print("📊 Both methods highlight similar regions, validating each other!")
print("   Occlusion is model-agnostic but much slower (O(n²) forward passes).")

---
## 10. Summary & Comparison of Interpretability Techniques

| Method | What it shows | Pros | Cons |
|--------|--------------|------|------|
| **Intermediate Activations** | Feature maps at each layer | Direct, no extra computation | Hard to interpret deep layers |
| **Filter Weights** | Raw learned convolution kernels | Simple, only for first layer | Only works for layer 1 (RGB input) |
| **Filter Visualization (Grad Ascent)** | What pattern maximally activates a filter | Works for any layer | Can produce abstract/noisy images |
| **Grad-CAM** | Which image regions drive classification | Fast, class-discriminative | Coarse resolution |
| **Guided Backpropagation** | Pixel-level feature attribution | High resolution | Not class-discriminative |
| **Guided Grad-CAM** | Combined high-res + class-specific | Best of both worlds | More computation |
| **Occlusion Sensitivity** | Importance of each region | Model-agnostic, intuitive | Very slow (many forward passes) |
| **t-SNE/PCA of Embeddings** | How network organizes representations | Global view of learned features | Requires many images |

### Key Takeaways

1. **ConvNets learn hierarchical features**: edges → textures → parts → objects
2. **Sparsity increases with depth**: deeper layers are more selective
3. **Grad-CAM is the most practical** interpretability tool for everyday use
4. **Multiple methods should be used together** to build a complete picture
5. **Different architectures** (VGG vs ResNet) attend to similar regions but with different precision

---
*Version 2 — with extensively detailed inline comments. Based on Chapter 10 of "Deep Learning with Python" by François Chollet.*